Модель для NMF

In [ ]:
import numpy as np
import librosa
import pickle
import os

TARGET_SR = 16000
N_FFT = 2048
HOP_LENGTH = 512
N_COMPONENTS_SPEECH = 80
N_COMPONENTS_MUSIC = 185
N_ITER = 1500
EPSILON = 1e-10

PROJECT_PATH = r'C:\Users\Dell\Desktop\myproject'
speech_path = os.path.join(PROJECT_PATH, 'data', 'clean_speech', 'speech_merged.wav')
music_path = os.path.join(PROJECT_PATH, 'data', 'clean_music', 'music_merged.wav')
model_path = os.path.join(PROJECT_PATH, 'models', 'nmf_model.pkl')

def get_spectrogram(audio, n_fft=N_FFT, hop_length=HOP_LENGTH):
    D = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    return np.abs(D) ** 2

# РЕАЛИЗАЦИЯ NMF
def train_nmf(V, n_components, n_iter=1500, random_state=42):
    """
    Обучает NMF на спектрограмме V, возвращает W и H.
    Используются мультипликативные правила обновления
    """
    np.random.seed(random_state)
    F, T = V.shape
    W = np.random.rand(F, n_components) + 0.1
    H = np.random.rand(n_components, T) + 0.1

    for i in range(n_iter):
        # Обновление H: H = H * (W^T V) / (W^T W H + eps)
        numerator_h = W.T @ V
        denominator_h = (W.T @ W) @ H + EPSILON
        H = H * (numerator_h / denominator_h)

        # Обновление W: W = W * (V H^T) / (W H H^T + eps)
        numerator_w = V @ H.T
        denominator_w = W @ (H @ H.T) + EPSILON
        W = W * (numerator_w / denominator_w)

        if (i + 1) % 500 == 0:
            error = np.linalg.norm(V - W @ H, 'fro')
            print(f"Итерация {i+1}/{n_iter}, ошибка: {error:.2f}")

    return W, H

# ЗАГРУЗКА ДАННЫХ
speech, sr = librosa.load(speech_path, sr=TARGET_SR, mono=True)
music, _ = librosa.load(music_path, sr=TARGET_SR, mono=True)

V_speech = get_spectrogram(speech)
V_music = get_spectrogram(music)

print(f"Речь: {V_speech.shape}, Музыка: {V_music.shape}")

# ОБУЧЕНИЕ РЕЧЕВЫХ БАЗИСОВ
print("Обучение речевых базисов...")
W_speech, H_speech = train_nmf(V_speech, N_COMPONENTS_SPEECH, N_ITER)

# ОБУЧЕНИЕ МУЗЫКАЛЬНЫХ БАЗИСОВ
print("Обучение музыкальных базисов...")
W_music, H_music = train_nmf(V_music, N_COMPONENTS_MUSIC, N_ITER)

print(f"W_speech: {W_speech.shape}, W_music: {W_music.shape}")
print(f"Сумма W_speech: {np.sum(W_speech):.2f}")
print(f"Сумма W_music: {np.sum(W_music):.2f}")

# СОХРАНЕНИЕ МОДЕЛИ
model_data = {
    'W_speech': W_speech,
    'W_music': W_music,
    'n_speech': N_COMPONENTS_SPEECH,
    'n_music': N_COMPONENTS_MUSIC,
    'target_sr': TARGET_SR,
    'n_fft': N_FFT,
    'hop_length': HOP_LENGTH,
    'n_freq_bins': W_speech.shape[0]
}

os.makedirs(os.path.dirname(model_path), exist_ok=True)
with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"Модель сохранена: {model_path}")

Алгоритм:
1. Загрузка обученной модели, содержащей матрицы спектральных базисов речи
2. Загрузка зашумленного аудиосигнала и вычисление его спектрограммы мощности с сохранением фазы для последующего восстановления
3. Объединение речевых и музыкальных базисов в единую матрицу
4. Поиск матрицы активаций H для зашумленного сигнала
5. Разделение матрицы активаций H на речевую (H_speech) и музыкальную (H_music) части
6. Реконструкция спектрограмм речи
7. Восстановление очищенного аудиосигнала во временной области с использованием обратного STFT и сохранённой фазы исходного сигнала
8. Нормализация громкости очищенного сигнала и сохранение результата

In [ ]:
import numpy as np
import librosa
import soundfile as sf
import pickle
import os

TARGET_SR = 16000
N_FFT = 2048
HOP_LENGTH = 512
EPSILON = 1e-10

PROJECT_PATH = r'C:\Users\Dell\Desktop\myproject'
model_path = os.path.join(PROJECT_PATH, 'models', 'nmf_model.pkl')
input_path = os.path.join(PROJECT_PATH, 'input', 'noisy_speech.wav')
output_path = os.path.join(PROJECT_PATH, 'output', 'cleaned_speech_nmf_only.wav')

def get_spectrogram(audio, n_fft=N_FFT, hop_length=HOP_LENGTH):
    D = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    power = np.abs(D) ** 2
    phase = np.angle(D)
    return power, phase

def reconstruct_audio(power, phase, hop_length=HOP_LENGTH):
    S = np.sqrt(power) * np.exp(1j * phase)
    return librosa.istft(S, hop_length=hop_length)

W = np.hstack([W_speech, W_music])

def solve_for_H(V, W, n_iter=300):
    """
    Находит H при фиксированном W (мультипликативное правило).
    Используется для разделения зашумленного сигнала.
    """
    F_V, T_V = V.shape
    F_W, K = W.shape

    if F_V != F_W:
        V = V[:F_W, :]

    H = np.random.rand(K, V.shape[1]) + 0.1

    for i in range(n_iter):
        numerator = W.T @ V
        denominator = (W.T @ W) @ H + EPSILON
        H = H * (numerator / denominator)

        if (i + 1) % 100 == 0:
            error = np.linalg.norm(V - W @ H, 'fro')
            print(f"  Итерация {i+1}/{n_iter}, ошибка: {error:.2f}")

    return H

# ЗАГРУЗКА МОДЕЛИ
with open(model_path, 'rb') as f:
    model = pickle.load(f)

W_speech = model['W_speech']
W_music = model['W_music']
n_speech = model['n_speech']
n_freq_bins = model['n_freq_bins']
n_fft = model['n_fft']
hop_length = model['hop_length']

# ЗАГРУЗКА ЗАШУМЛЕННОГО СИГНАЛА
noisy, sr = librosa.load(input_path, sr=TARGET_SR, mono=True)

V_noisy, phase = get_spectrogram(noisy, n_fft=n_fft, hop_length=hop_length)

if V_noisy.shape[0] != n_freq_bins:
    V_noisy = V_noisy[:n_freq_bins, :]
    phase = phase[:n_freq_bins, :]

# ПОИСК АКТИВАЦИЙ H
print("Поиск активаций H...")
H = solve_for_H(V_noisy, W, n_iter=300)

H_speech = H[:n_speech, :]
H_music = H[n_speech:, :]

V_speech_recon = W_speech @ H_speech
V_music_recon = W_music @ H_music

V_clean = V_speech_recon

# ВОССТАНОВЛЕНИЕ АУДИО
# Используем фазу от зашумленного сигнала
cleaned_audio = reconstruct_audio(V_clean, phase, hop_length=hop_length)

max_val = np.max(np.abs(cleaned_audio))
if max_val > 0:
    cleaned_audio = cleaned_audio / max_val

sf.write(output_path, cleaned_audio, TARGET_SR)

print(f"Готово: {output_path}")

Проверка метриками PESQ и STOI

In [ ]:
import librosa
from pesq import pesq

#Оценка качества звука
def calculate_pesq(reference_file, cleaned_file, target_sr=16000):

    try:
        ref_audio, sr_ref = librosa.load(reference_file, sr=target_sr)
        clean_audio, sr_deg = librosa.load(cleaned_file, sr=target_sr)
        
        min_len = min(len(ref_audio), len(clean_audio))
        ref_audio = ref_audio[:min_len]
        clean_audio = clean_audio[:min_len]
        
        mode = 'wb' if target_sr == 16000 else 'nb'
        
        # Вычисляем PESQ
        score = pesq(target_sr, ref_audio, clean_audio, mode)
        return score
        
    except Exception as e:
        print(f"Ошибка: {e}")
        return None
    
import librosa
import soundfile as sf
from pystoi import stoi
import numpy as np

#Оценка понятности речи 
def calculate_stoi(path_clean, path_denoised, target_sr=16000):
    
    try:
        clean_speech, fs = librosa.load(path_clean, sr=target_sr, mono=True)
        denoised_speech, _ = librosa.load(path_denoised, sr=target_sr, mono=True)

        min_len = min(len(clean_speech), len(denoised_speech))
        clean_speech = clean_speech[:min_len]
        denoised_speech = denoised_speech[:min_len]

        stoi_score = stoi(clean_speech, denoised_speech, fs, extended=False)
        return stoi_score
        
    except Exception as e:
        print(f"Ошибка: {e}")
        return None

if __name__ == "__main__":
    ref_path = r"\myproject\data\clean_speech.wav"
    clean_path = r"\myproject\output\cleaned_speech.wav" 
    
    score_pesq = calculate_pesq(ref_path, clean_path, target_sr=16000)
    
    if score_pesq is not None:
        print(f"Оценка PESQ: {score_pesq:.3f}")

    score_stoi = calculate_stoi(ref_path, clean_path, target_sr=16000)
    
    if score_stoi is not None:
        print(f"Оценка STOI: {score_stoi:.3f}")

Оценка PESQ: 1.391
Оценка STOI: 0.705
